# Collections recovery investigation

This notebook reconstructs recorded payment performance from the supplied synthetic data. It distinguishes gross SUCCESS events from net settled cash and avoids unsupported causal/ROI claims. Run from repository root.


In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
DATA = ROOT / 'output' / 'preprocessed'
GOLD = DATA / 'golden'
payments = pd.read_csv(GOLD / 'golden_payments.csv')
accounts = pd.read_csv(GOLD / 'golden_accounts_current.csv')


## Audit payment identity before summing

The source uses repeated references across different payment IDs/accounts, so `payment_reference` is not a global key. The build pipeline removes exact duplicate rows and coalesces a blank reference only when repeated `payment_id` rows otherwise agree.


In [ ]:
payments.payment_status.value_counts(dropna=False)


## Monthly recorded cash

Primary outcome is gross recorded amount where status is SUCCESS. No reversal linkage is supplied, and August is partial through 8 Aug. Thus the series tests the headline direction but does not establish a true recovery rate.


In [ ]:
p = payments.copy()
p['event_at'] = pd.to_datetime(p.event_at, errors='coerce')
s = p[p.payment_status.eq('SUCCESS')].copy()
s['month'] = s.event_at.dt.to_period('M').astype(str)
monthly = (s.groupby('month').agg(events=('payment_id','nunique'), gross_cash=('amount','sum'),
                               paying_accounts=('account_id','nunique')).reset_index())
monthly['mom_pct'] = monthly.gross_cash.pct_change()*100
monthly['month_complete'] = monthly.month.ne('2026-08')
monthly


## PTP and contact metric limitations

The source supplies KEPT/BROKEN PTP labels, so their ratio among closed labels can be reproduced but not independently validated against payment timing. Call status ANSWERED is not right-party contact (RPC); disposition codes vary across legacy/v1/v2 and lack an authoritative mapping. Avoid presenting RPC/contact rates as facts.


In [ ]:
ptp = pd.read_csv(DATA/'promises_to_pay.csv')
closed = ptp[ptp.status.isin(['KEPT','BROKEN'])]
ptp_proxy = (closed.status.eq('KEPT').mean(), len(closed))
ptp_proxy


## Decision and counterfactual

No rollout date, credible holdout, channel cost, or agent/vendor cost is available. Regression or before/after comparisons would confound selection and mix. Use a stratified randomized holdout for targeting (intention-to-treat, 30-day matured cash net of reversals), then calculate ROI from measured costs. Do not infer ₹10 Cr payback from this dataset.
